In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-24

@author:       Oscar Trevizo
@institution:  Harvard Extension School
@credential:   Graduate Certificate in Data Science (2023)
@context:      Independent project — applying graduate-level concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

UN WPP + World Bank — Two-Source Data Product Vignette
=======================================================

Description:
    Demonstrates how two governed data products are combined to form a third.
    DP1 (UN WPP demographics) and DP2 (World Bank GDP) are each wrapped
    independently with metadata, a semantic layer, and lineage tracking.
    Their merge produces DP3, whose lineage traces back through both parents.

    Join key : ISO3 (country code) + Year
    DP1 pipeline: load_un_wpp() -> filter_countries() -> clean_types()
    DP2 pipeline: load_wb_gdp()
    DP3:          inner join on ISO3 + Year; inherits both semantic layers

    Sources:
        UN WPP 2024     — https://population.un.org/wpp/
        World Bank WDI  — https://databank.worldbank.org/

    Support module: data_product_lib.py (same folder)

Key Concepts:
    Data Product Composition  — DP3 is built from DP1 and DP2, not from raw files
    Cross-source Semantic Layer — business names resolve across both sources
    Inherited Lineage         — DP3 carries all steps from DP1 and DP2
    Trustworthy Join          — join is expressed in semantic names, not column names

Revision History:
    2026-06-24  Original development
                - DP1: UN WPP 5-indicator panel, 1961-2023
                - DP2: World Bank GDP_USD + GDP_growth_pct, 1961-2023
                - DP3: inner join on ISO3 + Year; 5-step inherited lineage
"""

'\nCreated on 2026-06-24\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — applying course concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nUN WPP + World Bank — Two-Source Data Product Vignette\n=======================================================\n\nDescription:\n    Demonstrates how two governed data products are combined to form a third.\n    DP1 (UN WPP demographics) and DP2 (World Bank GDP) are each wrapped\n    independently with metadata, a semantic layer, and lineage tracking.\n    Their merge produces DP3, whose lineage traces back through both parents.\n\n    Join key : ISO3 (country code) + Year\n    DP1 pipeline: load_un_wpp() -> filter_countries() -> clean_types()\n    DP2 pipeline: load_wb_gdp()\n    DP3:          inner join on ISO3 + Year; inherits both semantic layers\n\n    Sources:\n        UN WPP 2024     — https://population.un

## Overview

A data product is more valuable when it can be composed with other data products.
This vignette demonstrates that composition pattern:

```
DP1  (UN WPP Demographics)         DP2  (World Bank GDP)
 └─ load_un_wpp()                   └─ load_wb_gdp()
 └─ filter_countries()                  ↓
 └─ clean_types()              ┌────────┴────────┐
        ↓                      │   inner join    │
        └──────────────────────┤  ISO3 + Year    ├──→  DP3  (Merged)
                               └─────────────────┘
```

DP3's lineage includes every step from both parents. Its semantic layer spans
both sources. A single contract JSON documents the full provenance chain.

| Product | Source | Indicators | Join key |
|---|---|---|---|
| DP1 | UN WPP 2024 | 5 demographic | ISO3 + Year |
| DP2 | World Bank WDI | 2 economic | ISO3 + Year |
| DP3 | DP1 ⊕ DP2 | 7 combined | — |

In [2]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from data_product_lib import (
    DataProductMetadata,
    SemanticLayer,
    LineageTracker,
    DataProduct,
)

---
## DP1 — UN World Population Prospects

Same three-step pipeline as `un_wpp_data_product.ipynb`. Reproduced here so
this notebook is self-contained.

In [3]:
UN_FILE    = '../../data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx'
YEAR_START = 1961
YEAR_END   = 2023


def load_un_wpp(filepath, year_start=YEAR_START, year_end=YEAR_END):
    df = pd.read_excel(
        filepath,
        sheet_name='Estimates',
        skiprows=16,
        thousands=' ',
        usecols=[
            'Region, subregion, country or area *',
            'ISO3 Alpha-code',
            'ISO2 Alpha-code',
            'Type',
            'Year',
            'Total Population, as of 1 July (thousands)',
            'Median Age, as of 1 July (years)',
            'Population Growth Rate (percentage)',
            'Total Fertility Rate (live births per woman)',
            'Life Expectancy at Birth, both sexes (years)',
            'Net Number of Migrants (thousands)',
            'Net Migration Rate (per 1,000 population)',
        ]
    )
    df = df.rename(columns={
        'Region, subregion, country or area *'         : 'Location',
        'ISO3 Alpha-code'                              : 'ISO3',
        'ISO2 Alpha-code'                              : 'ISO2',
        'Type'                                         : 'LocType',
        'Total Population, as of 1 July (thousands)'  : 'Population_Ks',
        'Median Age, as of 1 July (years)'             : 'MedAge',
        'Population Growth Rate (percentage)'          : 'PopulationGrowthRate',
        'Total Fertility Rate (live births per woman)' : 'FertilityRate_births_per_woman',
        'Life Expectancy at Birth, both sexes (years)' : 'LifeExpectancy',
        'Net Number of Migrants (thousands)'           : 'NetMigrants_Ks',
        'Net Migration Rate (per 1,000 population)'    : 'NetMigrationRate_per_Kpop',
    })
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(np.int64)
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    df.loc[df['LocType'] == 'Country/Area', 'LocType'] = 'Country'
    df['ReceivesMigrants']    = (df['NetMigrants_Ks'] > 0).astype(int)
    df['ImmigrantsEmigrants'] = df['NetMigrants_Ks'].apply(
        lambda x: 'Immigrants' if x > 0 else 'Emigrants'
    )
    return df.reset_index(drop=True)


def filter_countries(df):
    return df[df['LocType'] == 'Country'].reset_index(drop=True)


def clean_types(df):
    numeric_cols = [
        'Population_Ks', 'MedAge', 'PopulationGrowthRate',
        'FertilityRate_births_per_woman', 'LifeExpectancy',
        'NetMigrants_Ks', 'NetMigrationRate_per_Kpop',
    ]
    df = df.copy()
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [4]:
wpp_lineage = LineageTracker()

raw_df = load_un_wpp(UN_FILE)
wpp_lineage.log('1-load', 'load_un_wpp', (0, 0), raw_df.shape,
                f'Estimates sheet, skiprows=16, years {YEAR_START}–{YEAR_END}')

countries_df = filter_countries(raw_df)
wpp_lineage.log('2-filter', 'filter_countries', raw_df.shape, countries_df.shape,
                'LocType == Country only')

wpp_df = clean_types(countries_df)
wpp_lineage.log('3-clean', 'clean_types', countries_df.shape, wpp_df.shape,
                'pd.to_numeric on 7 numeric columns')

print(f'DP1 (WPP) : {wpp_df.shape}  |  '
      f'{wpp_df.Location.nunique()} countries  |  '
      f'{wpp_df.Year.min()}–{wpp_df.Year.max()}')

DP1 (WPP) : (14931, 14)  |  237 countries  |  1961–2023


In [5]:
wpp_metadata = DataProductMetadata(
    name              = 'UN WPP Demographics',
    description       = 'Country-year panel of UN WPP 2024 demographic indicators, 1961–2023',
    domain            = 'Demographics',
    owner             = 'Oscar Trevizo',
    source            = 'UN World Population Prospects 2024',
    source_url        = 'https://population.un.org/wpp/',
    license           = 'CC BY 3.0 IGO',
    version           = '1.0',
    refresh_frequency = 'Every 2 years (UN WPP release cycle)',
    created_at        = datetime.now(timezone.utc).isoformat(),
)

wpp_semantic = SemanticLayer()
wpp_semantic.register('net_migration_rate', 'NetMigrationRate_per_Kpop',
                       'per 1,000 population', 'Net migrants per 1,000 residents',
                       source_system='UN WPP 2024')
wpp_semantic.register('net_migrants',       'NetMigrants_Ks',
                       'thousands of people', 'Net number of migrants',
                       source_system='UN WPP 2024')
wpp_semantic.register('population',         'Population_Ks',
                       'thousands', 'Total population, as of 1 July',
                       source_system='UN WPP 2024')
wpp_semantic.register('fertility_rate',     'FertilityRate_births_per_woman',
                       'births per woman', 'Total fertility rate',
                       source_system='UN WPP 2024')
wpp_semantic.register('life_expectancy',    'LifeExpectancy',
                       'years', 'Life expectancy at birth',
                       source_system='UN WPP 2024')

dp_wpp = DataProduct(wpp_df, wpp_metadata, wpp_semantic, wpp_lineage)
dp_wpp.card()

  Data Product : UN WPP Demographics
  Description  : Country-year panel of UN WPP 2024 demographic indicators, 1961–2023
  Domain       : Demographics
  Owner        : Oscar Trevizo
  Source       : UN World Population Prospects 2024
  License      : CC BY 3.0 IGO
  Version      : 1.0
  Refresh      : Every 2 years (UN WPP release cycle)
  Created      : 2026-06-24T13:31:53.546471+00:00
  Rows         : 14,931
  Columns      : 14
  Semantic layer (5 entries):
    net_migration_rate             → NetMigrationRate_per_Kpop  [per 1,000 population]
    net_migrants                   → NetMigrants_Ks  [thousands of people]
    population                     → Population_Ks  [thousands]
    fertility_rate                 → FertilityRate_births_per_woman  [births per woman]
    life_expectancy                → LifeExpectancy  [years]
  Incomplete columns (<100%):
    ISO2                                       99.6%
  Lineage (3 steps):
    [1-load] load_un_wpp  (0, 0) → (18711, 14)  // Estim

---
## DP2 — World Bank GDP Indicators

Two indicators from the World Bank World Development Indicators:

| File | WB code | Column | Unit |
|---|---|---|---|
| `WB_GDP.xls` | `NY.GDP.MKTP.KD` | `GDP_USD` | Constant USD |
| `WB_GDP_growth.xls` | `NY.GDP.MKTP.KD.ZG` | `GDP_growth_pct` | % per year |

The WB files are wide-format (one column per year). `load_wb_gdp()` unpivots
them to long format and joins on `ISO3 + Year`. Region aggregate rows
(e.g. `WLD`, `AFE`) survive this step but are removed by the inner join
with DP1, which only contains country-level ISO3 codes.

In [6]:
WB_GDP_FILE        = '../../data/WB_GDP.xls'
WB_GDP_GROWTH_FILE = '../../data/WB_GDP_growth.xls'


def load_wb_gdp(gdp_file, gdp_growth_file, year_start=YEAR_START, year_end=YEAR_END):
    """
    Load World Bank GDP and GDP growth indicators from two XLS files.
    Each file is wide-format (year columns); stack() unpivots to long.
    Join key is ISO3 (Country Code) — more robust than country name.

    Parameters
    ----------
    gdp_file        : str, path to WB_GDP.xls (NY.GDP.MKTP.KD)
    gdp_growth_file : str, path to WB_GDP_growth.xls (NY.GDP.MKTP.KD.ZG)
    year_start      : int, first year inclusive. Default 1961.
    year_end        : int, last year inclusive.  Default 2023.

    Returns
    -------
    pd.DataFrame — long format, columns: ISO3, Year, GDP_USD, GDP_growth_pct
    """
    def _load_one(filepath, indicator):
        df = pd.read_excel(filepath, skiprows=3, sheet_name='Data')
        df = df.rename(columns={'Country Code': 'ISO3'})
        df = df.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code'])
        # Drop 2025 placeholder — all null
        df = df.drop(columns=[c for c in df.columns if str(c) == '2025'],
                     errors='ignore')
        # Wide -> long: stack year columns
        df = df.set_index('ISO3').stack().reset_index()
        df.columns = ['ISO3', 'Year', indicator]
        df['Year']    = df['Year'].astype(np.int64)
        df[indicator] = pd.to_numeric(df[indicator], errors='coerce')
        return df

    gdp_df        = _load_one(gdp_file,        'GDP_USD')
    gdp_growth_df = _load_one(gdp_growth_file, 'GDP_growth_pct')

    df = pd.merge(gdp_df, gdp_growth_df, on=['ISO3', 'Year'])
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]

    return df.reset_index(drop=True)

In [7]:
wb_lineage = LineageTracker()

wb_df = load_wb_gdp(WB_GDP_FILE, WB_GDP_GROWTH_FILE)
wb_lineage.log('1-load', 'load_wb_gdp', (0, 0), wb_df.shape,
               f'GDP_USD + GDP_growth_pct, wide→long, years {YEAR_START}–{YEAR_END}')

print(f'DP2 (WB)  : {wb_df.shape}  |  '
      f'{wb_df.ISO3.nunique()} ISO3 codes (incl. aggregates)  |  '
      f'{wb_df.Year.min()}–{wb_df.Year.max()}')
print()
print(wb_df.dtypes)
print()
wb_df.head()

DP2 (WB)  : (13805, 4)  |  259 ISO3 codes (incl. aggregates)  |  1961–2023

ISO3               object
Year                int64
GDP_USD           float64
GDP_growth_pct    float64
dtype: object



,ISO3,Year,GDP_USD,GDP_growth_pct
0,ABW,1987,1.123552e+09,16.078431
1,ABW,1988,1.333079e+09,18.648649
2,ABW,1989,1.494779e+09,12.129841
3,ABW,1990,1.553994e+09,3.961402
4,ABW,1991,1.677736e+09,7.962872


In [8]:
wb_metadata = DataProductMetadata(
    name              = 'World Bank GDP Indicators',
    description       = 'Country-year panel of World Bank GDP and growth indicators, 1961–2023',
    domain            = 'Economics',
    owner             = 'Oscar Trevizo',
    source            = 'World Bank World Development Indicators',
    source_url        = 'https://databank.worldbank.org/',
    license           = 'CC BY 4.0',
    version           = '1.0',
    refresh_frequency = 'Annual',
    created_at        = datetime.now(timezone.utc).isoformat(),
)

wb_semantic = SemanticLayer()
wb_semantic.register('gdp',        'GDP_USD',        'constant USD',
                      'GDP in constant US dollars (NY.GDP.MKTP.KD)',
                      source_system='World Bank WDI')
wb_semantic.register('gdp_growth', 'GDP_growth_pct', '% per year',
                      'GDP growth rate (NY.GDP.MKTP.KD.ZG)',
                      source_system='World Bank WDI')

dp_wb = DataProduct(wb_df, wb_metadata, wb_semantic, wb_lineage)
dp_wb.card()

  Data Product : World Bank GDP Indicators
  Description  : Country-year panel of World Bank GDP and growth indicators, 1961–2023
  Domain       : Economics
  Owner        : Oscar Trevizo
  Source       : World Bank World Development Indicators
  License      : CC BY 4.0
  Version      : 1.0
  Refresh      : Annual
  Created      : 2026-06-24T13:32:02.861599+00:00
  Rows         : 13,805
  Columns      : 4
  Semantic layer (2 entries):
    gdp                            → GDP_USD  [constant USD]
    gdp_growth                     → GDP_growth_pct  [% per year]
  All columns 100% complete.
  Lineage (1 steps):
    [1-load] load_wb_gdp  (0, 0) → (13805, 4)  // GDP_USD + GDP_growth_pct, wide→long, years 1961–2023


---
## DP3 — Merged Product (DP1 ⊕ DP2)

`merge_data_products()` performs three things at once:

1. **Inner join** on `ISO3 + Year` — keeps only countries and years present in
   both sources. Region aggregates in DP2 drop out here.
2. **Combined semantic layer** — all business names from DP1 and DP2 registered
   in one registry. The `source_system` field on each entry shows which source
   it came from.
3. **Inherited lineage** — every step from DP1 and DP2 is re-logged under
   `dp1/` and `dp2/` prefixes, then the merge step is appended. DP3's
   lineage is a complete audit trail from raw files to merged artifact.

In [ ]:
def merge_data_products(dp1, dp2, on):
    """
    Inner join two DataProducts on shared key columns.
    Combines their semantic layers and inherits both lineage trails.

    Raises ValueError if both products share a business name in their
    semantic layers — prevents silent overwrite of one source's definition
    by another.

    Parameters
    ----------
    dp1 : DataProduct — left (primary) product
    dp2 : DataProduct — right product
    on  : list[str]  — shared column names to join on

    Returns
    -------
    (merged_df, merged_semantic, merged_lineage)
    Caller wraps these into a new DataProduct with its own metadata.
    """
    # Collision check — fail fast rather than silently shadow a definition
    dp1_names   = set(dp1.semantic_layer.to_dict().keys())
    dp2_names   = set(dp2.semantic_layer.to_dict().keys())
    collisions  = dp1_names & dp2_names
    if collisions:
        raise ValueError(
            f'Semantic name collision(s) between DP1 and DP2: {sorted(collisions)}. '
            'Rename the conflicting entries in one source before merging.'
        )

    merged_df = pd.merge(dp1.data, dp2.data, on=on, how='inner')

    # Combined semantic layer — source_system distinguishes provenance
    merged_semantic = SemanticLayer()
    for name, entry in dp1.semantic_layer.to_dict().items():
        merged_semantic.register(name, entry['column'], entry['unit'],
                                 entry['description'], entry['source_system'])
    for name, entry in dp2.semantic_layer.to_dict().items():
        merged_semantic.register(name, entry['column'], entry['unit'],
                                 entry['description'], entry['source_system'])

    # Inherit all lineage steps from both parents, then log the merge
    merged_lineage = LineageTracker()
    for s in dp1.lineage.to_list():
        merged_lineage.log(f"dp1/{s['step']}", s['operation'],
                           s['input_shape'], s['output_shape'], s['notes'])
    for s in dp2.lineage.to_list():
        merged_lineage.log(f"dp2/{s['step']}", s['operation'],
                           s['input_shape'], s['output_shape'], s['notes'])
    merged_lineage.log(
        'merge', 'inner_join',
        dp1.data.shape, merged_df.shape,
        f'keys: {"+".join(on)}  |  DP2 input: {dp2.data.shape}',
    )

    return merged_df, merged_semantic, merged_lineage

In [10]:
merged_df, merged_semantic, merged_lineage = merge_data_products(
    dp_wpp, dp_wb, on=['ISO3', 'Year']
)

merged_metadata = DataProductMetadata(
    name              = 'UN WPP + World Bank — Demographics and GDP',
    description       = ('Country-year panel merging UN WPP demographics '
                         'and World Bank GDP indicators, 1961–2023'),
    domain            = 'Demographics / Economics',
    owner             = 'Oscar Trevizo',
    source            = 'UN WPP 2024 + World Bank WDI',
    source_url        = 'https://population.un.org/wpp/ | https://databank.worldbank.org/',
    license           = 'CC BY 3.0 IGO (WPP) + CC BY 4.0 (WB)',
    version           = '1.0',
    refresh_frequency = 'Annual (WB) / Every 2 years (WPP)',
    created_at        = datetime.now(timezone.utc).isoformat(),
)

dp_merged = DataProduct(merged_df, merged_metadata, merged_semantic, merged_lineage)

n_wpp_steps = len(dp_wpp.lineage.to_list())
n_wb_steps  = len(dp_wb.lineage.to_list())
print(f'DP3 shape   : {dp_merged.data.shape}')
print(f'Lineage     : {len(dp_merged.lineage.to_list())} steps  '
      f'({n_wpp_steps} WPP + {n_wb_steps} WB + 1 merge)')
print(f'Semantic    : {len(dp_merged.semantic_layer.to_dict())} entries  '
      f'(5 WPP + 2 WB)')

DP3 shape   : (10920, 16)
Lineage     : 5 steps  (3 WPP + 1 WB + 1 merge)
Semantic    : 7 entries  (5 WPP + 2 WB)


In [11]:
dp_merged.card()

  Data Product : UN WPP + World Bank — Demographics and GDP
  Description  : Country-year panel merging UN WPP demographics and World Bank GDP indicators, 1961–2023
  Domain       : Demographics / Economics
  Owner        : Oscar Trevizo
  Source       : UN WPP 2024 + World Bank WDI
  License      : CC BY 3.0 IGO (WPP) + CC BY 4.0 (WB)
  Version      : 1.0
  Refresh      : Annual (WB) / Every 2 years (WPP)
  Created      : 2026-06-24T13:32:12.503679+00:00
  Rows         : 10,920
  Columns      : 16
  Semantic layer (7 entries):
    net_migration_rate             → NetMigrationRate_per_Kpop  [per 1,000 population]
    net_migrants                   → NetMigrants_Ks  [thousands of people]
    population                     → Population_Ks  [thousands]
    fertility_rate                 → FertilityRate_births_per_woman  [births per woman]
    life_expectancy                → LifeExpectancy  [years]
    gdp                            → GDP_USD  [constant USD]
    gdp_growth                

---
## Cross-Source Query via Semantic Layer

The merged semantic layer resolves business names from **both** sources without
the caller knowing which underlying column belongs to which file.
If a future WPP release renames `NetMigrationRate_per_Kpop`, only the
`register()` call needs updating — the query below keeps working.

In [12]:
# Resolve column names from business names — each from a different source
mig_entry = dp_merged.semantic_layer.get('net_migration_rate')
gdp_entry = dp_merged.semantic_layer.get('gdp_growth')

mig_col = mig_entry['column']   # NetMigrationRate_per_Kpop  (UN WPP)
gdp_col = gdp_entry['column']   # GDP_growth_pct              (World Bank)

print(f"net_migration_rate  →  {mig_col:<35s} [{mig_entry['source_system']}]")
print(f"gdp_growth          →  {gdp_col:<35s} [{gdp_entry['source_system']}]")
print()

# Top 10 net-receiving countries in 2020 with their GDP growth that year
top10 = (
    dp_merged.data[['Location', 'ISO3', 'Year', mig_col, gdp_col]]
    .query('Year == 2020')
    .nlargest(10, mig_col)
    .reset_index(drop=True)
)
top10.columns = [
    'Location', 'ISO3', 'Year',
    f'net_migration_rate ({mig_entry["unit"]})',
    f'gdp_growth ({gdp_entry["unit"]})',
]
top10

net_migration_rate  →  NetMigrationRate_per_Kpop           [UN WPP 2024]
gdp_growth          →  GDP_growth_pct                      [World Bank WDI]



,Location,ISO3,Year,"net_migration_rate (per 1,000 population)",gdp_growth (% per year)
0,Monaco,MCO,2020,26.255,-13.239105
1,Maldives,MDV,2020,19.203,-32.908829
2,Turks and Caicos Islands,TCA,2020,15.365,-33.817097
3,Seychelles,SYC,2020,14.349,-11.739861
4,Slovenia,SVN,2020,13.562,-4.085001
5,Malta,MLT,2020,13.323,-3.457283
6,"China, Macao SAR",MAC,2020,12.945,-54.402093
7,Syrian Arab Republic,SYR,2020,12.782,-0.697155
8,Sint Maarten (Dutch part),SXM,2020,12.656,-13.316789
9,Cayman Islands,CYM,2020,11.793,-4.954807


In [13]:
# Full lineage of DP3 — every step from both parents
pd.DataFrame(dp_merged.lineage.to_list())[['step', 'operation', 'input_shape',
                                            'output_shape', 'notes']]

,step,operation,input_shape,output_shape,notes
0,dp1/1-load,load_un_wpp,"(0, 0)","(18711, 14)","Estimates sheet, skiprows=16, years 1961–2023"
1,dp1/2-filter,filter_countries,"(18711, 14)","(14931, 14)",LocType == Country only
2,dp1/3-clean,clean_types,"(14931, 14)","(14931, 14)",pd.to_numeric on 7 numeric columns
3,dp2/1-load,load_wb_gdp,"(0, 0)","(13805, 4)","GDP_USD + GDP_growth_pct, wide→long, years 196..."
4,merge,inner_join,"(14931, 14)","(10920, 16)","keys: ISO3+Year | DP2 input: (13805, 4)"


---
## Export

Two artifacts for DP3 only — DP1 and DP2 have their own contracts in
`un_wpp_data_product.ipynb`. The DP3 contract JSON embeds the full
5-step lineage so downstream consumers can trace provenance without
opening any of the source notebooks.

In [14]:
import os

out_dir = 'output'
os.makedirs(out_dir, exist_ok=True)

parquet_path  = f'{out_dir}/un_wpp_wb_merged.parquet'
contract_path = f'{out_dir}/un_wpp_wb_merged_contract.json'

dp_merged.data.to_parquet(parquet_path, index=False)
print(f'Data written     : {parquet_path}  '
      f'({os.path.getsize(parquet_path):,} bytes)')

dp_merged.contract(contract_path)

Data written     : output/un_wpp_wb_merged.parquet  (656,090 bytes)
Contract written: output/un_wpp_wb_merged_contract.json


---
## What's Next

DP3 demonstrates data product composition: a merge is not just a `pd.merge()` —
it carries the full lineage of its parents, a combined semantic layer that spans
both sources, and a single contract JSON that documents the entire provenance
chain from raw files to merged artifact.

**Possible extensions from here:**

- **DP4 = DP3 ⊕ DP_new** — add a third source (e.g. UN migration stock,
  US DHS naturalization data) and watch the lineage deepen automatically.
- **Harvard project replication** — expand the World Bank panel to 15 indicators
  (food, homicides, electricity, CO₂, education, CPI, migration stock) and build
  the regression / tree models from `File3` onward — this time in Python.
  That project (STAT E-109, Spring 2023) is the modeling counterpart to this
  data product series.
- **Outlier removal as a logged step** — add a z-score filter on
  `net_migration_rate` between DP1 and DP3, logged as a lineage step,
  showing that cleaning decisions are auditable and not hidden.